# Getting Started

Build a galaxy, GMC, and interstellar-object model and integrate it.

## Potentials

The galaxy is a stationary background, while each GMC carries a potential as it moves. The compatible integrators currently evaluate their kernel-local unit Kepler force rather than the potentials in this model, so the example uses matching Kepler definitions until model-driven force dispatch is implemented.

In [ ]:
import drift as dft
import numpy as np

galaxy_potential = dft.Potential.kepler(amp=1.0)
gmc_potential = dft.Potential.kepler(amp=1.0)

## Containers

Initial states are `(N, 6)` float64 arrays with phase-space columns `[x, y, z, vx, vy, vz]`. A GMC is a potential-bearing particle group; ISOs are test particles and do not contribute a potential.

In [ ]:
galaxy = dft.background(galaxy_potential)
gmc = dft.particles(
    gmc_potential,
    np.array([[1.0, 0.0, 0.0, 0.0, 1.0, 0.0]], dtype=np.float64),
)
iso = dft.test_particles(
    np.array([[-1.0, 0.0, 0.0, 0.0, -1.0, 0.0]], dtype=np.float64)
)

## Configuration

`Config` defaults to the CPU DOPR54 compatible implementation. `add(node, *requires)` registers a particle group and its intended force sources: the GMC depends on the galaxy, while the ISO depends on both.

In [ ]:
sim = dft.Config(ts=(0.0, 2.0 * np.pi, 101))
sim.add(gmc, galaxy)
sim.add(iso, gmc, galaxy)

## Integrate

Results follow first-registration order. Particle trajectories have shape `(time, particle, component)`, while the stationary galaxy contributes `None`.

In [ ]:
gmc_trajectory, galaxy_result, iso_trajectory = sim.run()

assert gmc_trajectory is not None
assert gmc_trajectory.shape == (101, 1, 6)
assert galaxy_result is None
assert iso_trajectory is not None
assert iso_trajectory.shape == (101, 1, 6)

print("GMC:", gmc_trajectory.shape)
print("galaxy:", galaxy_result)
print("ISO:", iso_trajectory.shape)

## Next steps

See the API docstrings, for example `help(dft.Config)`, and the [testing instructions](../docs/Testing_Instructions.md) for backend and fixture commands.